# SSBU VOD helper

Insert your URL and click _Run after_ (Ctrl + F10 OR Runtime -> Run after) to get your output.

Note: please use either Youtube OR Twitch at once.


## Dependencies

Imports dependencies.


In [ ]:
# jupyter setup
import sys

sys.path.insert(0, "./src/")

%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import os.path
import glob
from pathlib import Path
import subprocess

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from tensorflow.keras.models import load_model  # type: ignore

from utils.vod_utils import (
    predict_screen,
    video_to_frames,
    setup_dirs,
    apply_thresholding,
    group_frames,
    plot_1d_scatter,
    extract_vod_timestamps,
    frames_to_games,
    games_to_sets,
)

In [ ]:
# load model from directory
model = load_model("./model")

# source url/directory (only use youtube or twitch at one time)
youtube_url = ""
twitch_url = "https://www.twitch.tv/videos/2257511253"

### Download from YouTube


In [ ]:
try:
    import pytube
except ModuleNotFoundError:
    raise ModuleNotFoundError("Module pytube not found: install it and try again.")

if youtube_url:
    yt = pytube.YouTube(youtube_url.replace("watch?v=", ""))
    yt = (
        yt.streams.filter(progressive=True, file_extension="mp4")
        .order_by("resolution")
        .desc()
        .first()
    )

    if not os.path.exists("."):
        os.makedirs(".")

    if yt:
        yt.download(".")

### Download from Twitch


In [ ]:
import importlib
import importlib.util

if twitch_url:
    try:
        if importlib.util.find_spec("twitchdl"):
            subprocess.run(
                ["twitch-dl", "download", "-q", "160p", "-f", "mp4", twitch_url]
            )

    except ModuleNotFoundError:
        raise ModuleNotFoundError(
            "Module twitchdl not found: install it and try again."
        )

## Directories handling

Creates snapshots folder and video-specific subfolders starting from the last added (downloaded) file.


In [ ]:
if twitch_url or youtube_url:
    # orders files by last added in order to grab last .mp4 file
    files = list(filter(os.path.isfile, glob.glob("*")))
    files.sort(key=lambda x: ".mp4" in x and os.path.getmtime(x))

    # handle situation in which no file is present
    if not files:
        raise Exception("An error has occurred: no video files in directory.")

    # get latest added file and set directory name
    input_file = files[-1]
    dir_name = input_file.replace(".mp4", "")
    vod_name = input_file[:21]

In [ ]:
input_file = Path(input_file)
dataframes_dir, timestamps_dir, snapshots_dir = setup_dirs(vod_name)

## Video to frames conversion

Uses `ffmpeg-python` to convert a video file to a succession of frames from the same video, grabbing one frame each second and resizing it to 426x240. **This process will take a lot of time** depending on the length of the video, so sit tight.


In [ ]:
if not len(glob.glob(f"{snapshots_dir}/**")) > 0:
    print(f"Processing: {input_file}")
    video_to_frames(input_file, snapshots_dir)
else:
    print(f"Snapshots already exist for {input_file}; skipping.")

## Frames prediction


In [ ]:
labels = {0: "gameplay", 1: "junk"}

snapshots = glob.glob(f"{snapshots_dir}/**")

predictions = [predict_screen(model, snap, labels) for snap in snapshots]

In [ ]:
df = frames_to_games(predictions)
df.to_csv(f"{dataframes_dir}/predictions.csv", index=False)

df

## Handling predictions


In [ ]:
df = pd.read_csv(f"{dataframes_dir}/predictions.csv")
df

In [ ]:
# save number of rows to measure thresholding later
old_n_rows = df.shape[0]

# remove every row with duration less than the threshold
df = apply_thresholding(df, "duration", threshold=15)

print(f"{old_n_rows - df.shape[0]} rows removed by thresholding.")

df

In [ ]:
group_df = group_frames(df)
group_df.to_csv(f"{dataframes_dir}/games.csv")

group_df

In [ ]:
fig = plt.figure(figsize=(17, 2))

date_formatter = mdates.DateFormatter("%H:%M:%S")
plt.gca().xaxis.set_major_formatter(date_formatter)

# plot_1d_scatter(df[df["label"] == "junk"], "timestamp", "junk")
# plot_1d_scatter(df[df["label"] == "gameplay"], "timestamp", "gameplay")

if "level_0" not in df.columns:
    df = df.reset_index()
plot_1d_scatter(df[df["label"] == "junk"], "index", "junk")
plot_1d_scatter(df[df["label"] == "gameplay"], "index", "gameplay", s=5)

plt.yticks([])
plt.xticks([])

plt.legend()

## Clustering


In [ ]:
games_df = pd.read_csv(
    f"{dataframes_dir}/games.csv", parse_dates=["timestamp", "end_ts"]
)
games_df

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score


scores = []
num_clusters = 18
tmp = games_df[games_df["label"] == "gameplay"]
tmp = tmp[["timestamp", "end_ts"]]

for n in range(2, num_clusters):
    agg = AgglomerativeClustering(
        n_clusters=n,
        linkage="complete",
    ).fit(tmp)

    labels = agg.labels_
    scores.append(silhouette_score(tmp, labels, metric="euclidean"))

plt.plot([i for i in range(2, num_clusters)], scores)

In [ ]:
n_clusters = 16
sets_df, model = games_to_sets(games_df, n_clusters=n_clusters)

# plot clustering
clusters_labels = [sets_df[sets_df["cluster"] == i] for i in range(n_clusters)]

fig = plt.figure(figsize=(15, 2))

date_formatter = mdates.DateFormatter("%H:%M:%S")
plt.gca().xaxis.set_major_formatter(date_formatter)

plt.yticks([])

plot_1d_scatter(sets_df, "timestamp", c="cluster", edgecolors="black")

extract_vod_timestamps(
    sets_df,
    f"{timestamps_dir}/{vod_name}.md",
    buffer=7,
    override=True,
)

In [ ]:
from utils.vod_utils import plot_dendrogram

plt.figure(figsize=(15, 5))
plot_dendrogram(model)